# Batter Regression to the Mean

## Goal
For each batter, compute their season-long average Daily_Value as a true-talent baseline, then track how their rolling 30-game performance drifts above and below that line throughout the season.

- **Baseline**: full-season mean per player (the regression-to-mean target)
- **Rolling window**: 30 games played (near-optimal per `analyze_roster_churn.ipynb`)
- **Deviation**: `rolling_30g_avg − season_mean` — positive = hot streak, negative = slump
- **Charts**: small-multiple deviation curves + league-wide heatmap

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

%matplotlib inline
plt.style.use('fivethirtyeight')

ROLLING_WINDOW = 30  # games played — from analyze_roster_churn optimal

In [ ]:
# 1. Load Data
SEASON = 2025

try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

PROJECT_ROOT = os.path.dirname(SCRIPT_DIR)
BASE_PATH = os.path.join(PROJECT_ROOT, 'data-lake', '01_Bronze', 'fantasy_baseball')

stats_df     = pd.read_csv(os.path.join(BASE_PATH, f'stats_mlb_daily_{SEASON}.csv'))
player_map_df = pd.read_csv(os.path.join(BASE_PATH, 'player_map.csv'))

stats_df['date'] = pd.to_datetime(stats_df['date'])
stats_df['playerId'] = stats_df['playerId'].astype(str)
player_map_df['espn_player_id'] = player_map_df['espn_player_id'].astype(str)
player_map_df['statcast_player_id'] = (
    player_map_df['statcast_player_id']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
)

stats_merged = stats_df.merge(
    player_map_df, left_on='playerId', right_on='statcast_player_id', how='left'
)
print(f'Merged shape: {stats_merged.shape}')

In [ ]:
# 2. Compute Batter Daily Z-Score Value
# Same method as analyze_roster_churn: per-day Z-scores among batters only.

batter_stats = ['R', 'HR', 'RBI', 'SB']

def _add_daily_zscore(df, col, mask, sign=1.0):
    if col not in df.columns:
        return
    sub = df.loc[mask]
    date_mean = sub.groupby('date')[col].transform('mean')
    date_std  = sub.groupby('date')[col].transform('std').replace(0, np.nan).fillna(1)
    z = ((sub[col] - date_mean) / date_std).fillna(0)
    df.loc[mask, 'Daily_Value'] += sign * z

daily = stats_merged.copy()
daily['Daily_Value'] = 0.0
mask_b = daily['b_or_p'] == 'batter'

for col in batter_stats:
    _add_daily_zscore(daily, col, mask_b)

daily = daily[mask_b].copy()
print(f'Batter rows: {len(daily):,}')
daily[['date', 'playerName', 'Daily_Value']].head()

In [ ]:
# 3. Aggregate to One Row per Player per Date (handle doubleheaders)
player_daily = (
    daily[['date', 'espn_player_id', 'playerName', 'Daily_Value']]
    .groupby(['date', 'espn_player_id', 'playerName'])
    .sum()
    .reset_index()
)

# Keep only days the player actually appeared
player_daily = player_daily.dropna(subset=['Daily_Value'])

# Per-player game index (0, 1, 2 … across the season)
player_daily = player_daily.sort_values(['espn_player_id', 'date'])
player_daily['game_idx'] = player_daily.groupby('espn_player_id').cumcount()

# Games played per player — used to filter to players with enough sample
games_played = player_daily.groupby('espn_player_id')['game_idx'].max().rename('games_played')
player_daily = player_daily.join(games_played, on='espn_player_id')

print(f'Unique batters: {player_daily["espn_player_id"].nunique()}')
print(f'Median games played: {games_played.median():.0f}')

In [ ]:
# 4. Compute Season Baseline and Rolling Deviation
MIN_GAMES = 60  # minimum games to be included — enough for a stable mean

qualified = player_daily[player_daily['games_played'] >= MIN_GAMES].copy()

# Season mean per player = regression-to-mean target
season_mean = qualified.groupby('espn_player_id')['Daily_Value'].mean().rename('season_mean')
qualified = qualified.join(season_mean, on='espn_player_id')

# Rolling 30-game average (over games played, not calendar days)
qualified['rolling_avg'] = (
    qualified
    .groupby('espn_player_id')['Daily_Value']
    .transform(lambda x: x.rolling(ROLLING_WINDOW, min_periods=10).mean())
)

# Deviation from season baseline
qualified['deviation'] = qualified['rolling_avg'] - qualified['season_mean']

print(f'Qualified batters (≥{MIN_GAMES} games): {qualified["espn_player_id"].nunique()}')
qualified[['playerName', 'game_idx', 'season_mean', 'rolling_avg', 'deviation']].head(10)

In [ ]:
# 5. Select Top Players by Season Total Daily_Value for Plotting
TOP_N = 12

top_players = (
    qualified.groupby(['espn_player_id', 'playerName'])['Daily_Value']
    .sum()
    .reset_index()
    .sort_values('Daily_Value', ascending=False)
    .head(TOP_N)
)
top_ids = top_players['espn_player_id'].tolist()

plot_df = qualified[qualified['espn_player_id'].isin(top_ids)].copy()

# Map id → name for labels
id_to_name = (
    qualified.drop_duplicates('espn_player_id')
    .set_index('espn_player_id')['playerName']
    .to_dict()
)

print('Top players selected:')
print(top_players[['playerName', 'Daily_Value']].to_string(index=False))

In [ ]:
# 6. Small-Multiple Deviation Curves
# Each panel: one player's rolling deviation from their season mean.
# Green fill = above baseline (hot), red fill = below (slump).

ncols = 3
nrows = int(np.ceil(TOP_N / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 3), sharey=False)
axes = axes.flatten()

for i, pid in enumerate(top_ids):
    ax = axes[i]
    pdata = plot_df[plot_df['espn_player_id'] == pid].dropna(subset=['deviation'])
    x = pdata['game_idx'].values
    y = pdata['deviation'].values

    ax.axhline(0, color='black', linewidth=1.0, linestyle='--', alpha=0.6)
    ax.plot(x, y, color='dimgray', linewidth=1.2)
    ax.fill_between(x, y, 0, where=(y >= 0), color='seagreen', alpha=0.35, label='Above avg')
    ax.fill_between(x, y, 0, where=(y <  0), color='firebrick', alpha=0.35, label='Below avg')

    name = id_to_name.get(pid, pid)
    smean = pdata['season_mean'].iloc[0]
    ax.set_title(f'{name}  (μ={smean:.2f})', fontsize=9)
    ax.set_xlabel('Game #', fontsize=8)
    ax.set_ylabel('Deviation', fontsize=8)
    ax.tick_params(labelsize=7)

# Hide unused panels
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    f'Rolling {ROLLING_WINDOW}-Game Deviation from Season Mean — Top {TOP_N} Batters ({SEASON})',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# 7. League-Wide Deviation Heatmap
# Rows = players (all qualified), columns = game index, colour = deviation from season mean.
# Reveals whether hot/cold streaks are correlated across the league (schedule effects, weather)
# or truly player-specific.

HEATMAP_PLAYERS = 40  # show top N by season total

heatmap_ids = (
    qualified.groupby('espn_player_id')['Daily_Value']
    .sum()
    .nlargest(HEATMAP_PLAYERS)
    .index.tolist()
)

heatmap_df = qualified[qualified['espn_player_id'].isin(heatmap_ids)]

pivot_heat = (
    heatmap_df
    .pivot_table(index='espn_player_id', columns='game_idx', values='deviation', aggfunc='mean')
)

# Sort rows by season mean descending so best players are at top
row_order = (
    qualified[qualified['espn_player_id'].isin(heatmap_ids)]
    .groupby('espn_player_id')['season_mean']
    .mean()
    .sort_values(ascending=False)
    .index
)
pivot_heat = pivot_heat.loc[row_order]
pivot_heat.index = [id_to_name.get(pid, pid) for pid in pivot_heat.index]

vmax = pivot_heat.abs().quantile(0.95).max()  # clip colour scale at 95th pct

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    pivot_heat,
    cmap='RdYlGn',
    center=0,
    vmin=-vmax, vmax=vmax,
    linewidths=0,
    ax=ax,
    cbar_kws={'label': 'Deviation from Season Mean'},
    yticklabels=True,
    xticklabels=10,
)
ax.set_title(
    f'Rolling {ROLLING_WINDOW}-Game Deviation Heatmap — Top {HEATMAP_PLAYERS} Batters ({SEASON})',
    fontsize=12
)
ax.set_xlabel('Game # (season progression →)')
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 9. Streak Length Distribution
# How long do hot vs cold streaks typically last?

streak_lengths = (
    streak_records
    .drop_duplicates('streak_id')[['streak_id', 'streak_type', 'streak_length']]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, stype, color in zip(axes, ['hot', 'cold'], ['seagreen', 'firebrick']):
    lengths = streak_lengths[streak_lengths['streak_type'] == stype]['streak_length']
    ax.hist(lengths, bins=range(MIN_STREAK_LEN, lengths.max() + 2), color=color, alpha=0.75, edgecolor='white')
    ax.axvline(lengths.median(), color='black', linestyle='--', linewidth=1.2,
               label=f'Median: {lengths.median():.0f}g')
    ax.axvline(lengths.mean(),   color='black', linestyle=':',  linewidth=1.2,
               label=f'Mean: {lengths.mean():.1f}g')
    ax.set_title(f'{"Hot" if stype == "hot" else "Cold"} Streak Lengths  (n={len(lengths)})')
    ax.set_xlabel('Streak Length (games)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

fig.suptitle('Distribution of Streak Lengths — Qualified Batters (2025)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 11. Event Study — Average Deviation Trajectory Around Streak Start vs End
# Align all streak starts to t=0, compute average deviation from t=-10 to t=+20.
# Do the same anchored on the last game of each streak (t=0 = final game).
# Shows the characteristic "shape" of a streak entering and exiting.

LOOKBACK  = 10
LOOKAHEAD = 20

def build_event_windows(streak_records, qualified, anchor='start'):
    """
    For each streak, extract a window of deviation values centred on the anchor game.
    anchor='start' -> anchored on pos_in_streak == 1
    anchor='end'   -> anchored on pos_in_streak == streak_length
    """
    dev_lookup = qualified.set_index(['espn_player_id', 'game_idx'])['deviation']
    windows = []

    for sid, grp in streak_records.groupby('streak_id'):
        grp = grp.sort_values('pos_in_streak')
        stype = grp['streak_type'].iloc[0]
        pid   = grp['espn_player_id'].iloc[0]

        if anchor == 'start':
            anchor_game = grp[grp['pos_in_streak'] == 1]['game_idx'].iloc[0]
        else:
            anchor_game = grp[grp['pos_in_streak'] == grp['streak_length'].max()]['game_idx'].iloc[0]

        row = {'streak_id': sid, 'streak_type': stype}
        for t in range(-LOOKBACK, LOOKAHEAD + 1):
            g = anchor_game + t
            try:
                row[t] = dev_lookup.loc[(pid, g)]
            except KeyError:
                row[t] = np.nan
        windows.append(row)

    return pd.DataFrame(windows)

start_windows = build_event_windows(streak_records, qualified, anchor='start')
end_windows   = build_event_windows(streak_records, qualified, anchor='end')

t_vals = range(-LOOKBACK, LOOKAHEAD + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, windows, title in [
    (axes[0], start_windows, 'Anchored on Streak START (t=0 = first game)'),
    (axes[1], end_windows,   'Anchored on Streak END   (t=0 = last game)'),
]:
    for stype, color, label in [('hot', 'seagreen', 'Hot'), ('cold', 'firebrick', 'Cold')]:
        sub = windows[windows['streak_type'] == stype][list(t_vals)]
        mean_traj = sub.mean()
        sem_traj  = sub.sem()
        ax.plot(t_vals, mean_traj.values, color=color, linewidth=2, label=label)
        ax.fill_between(
            t_vals,
            (mean_traj - sem_traj).values,
            (mean_traj + sem_traj).values,
            color=color, alpha=0.15
        )

    ax.axvline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.7, label='Anchor (t=0)')
    ax.axhline(0, color='black', linewidth=0.7, linestyle=':',  alpha=0.5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Games relative to anchor')
    ax.set_ylabel('Mean deviation from season avg')
    ax.legend(fontsize=8)

fig.suptitle('Event Study: Deviation Trajectory Around Streak Start vs End', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 15. Current Watchlist — Players in Active Ramp Phase (end of dataset)
# Players whose most recent game shows a hot or cold ramp signal.
# These are the players to watch on the waiver wire right now.

latest_game = qualified.groupby('espn_player_id')['game_idx'].max().reset_index()
latest_stats = qualified.merge(latest_game, on=['espn_player_id', 'game_idx'])

id_to_name = qualified.drop_duplicates('espn_player_id').set_index('espn_player_id')['playerName']

def make_watchlist(df, ramp_col, label):
    watch = df[df[ramp_col]][
        ['espn_player_id', 'game_idx', 'deviation', 'dev_slope', 'season_mean', 'rolling_avg']
    ].copy()
    watch['playerName'] = watch['espn_player_id'].map(id_to_name)
    watch = watch.sort_values('dev_slope', ascending=(label == 'Cold'))
    watch = watch[['playerName', 'game_idx', 'season_mean', 'rolling_avg', 'deviation', 'dev_slope']]
    watch.columns = ['Player', 'Games Played', 'Season Mean', 'Rolling Avg', 'Deviation', 'Slope/Game']
    return watch.reset_index(drop=True)

hot_watch  = make_watchlist(latest_stats, 'hot_ramp',  'Hot')
cold_watch = make_watchlist(latest_stats, 'cold_ramp', 'Cold')

print(f"=== HOT RAMP WATCHLIST ({len(hot_watch)} players) ===")
print(hot_watch.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f"\n=== COLD RAMP WATCHLIST ({len(cold_watch)} players) ===")
print(cold_watch.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

In [ ]:
# 14. Event Study — Trajectory After a Ramp Signal
# Align all ramp signal games to t=0 and plot average deviation from t=-5 to t=+15.
# If the signal is real, hot ramp should show sustained upward movement after t=0.

RAMP_LOOKBACK  = 5
RAMP_LOOKAHEAD = 15

def build_ramp_windows(ramp_col, qualified):
    dev_lookup = qualified.set_index(['espn_player_id', 'game_idx'])['deviation']
    windows = []
    signals = qualified[qualified[ramp_col]][['espn_player_id', 'game_idx']]
    for _, row in signals.iterrows():
        pid, g0 = row['espn_player_id'], row['game_idx']
        entry = {'espn_player_id': pid, 'anchor_game': g0}
        for t in range(-RAMP_LOOKBACK, RAMP_LOOKAHEAD + 1):
            try:    entry[t] = dev_lookup.loc[(pid, g0 + t)]
            except: entry[t] = np.nan
        windows.append(entry)
    return pd.DataFrame(windows)

hot_ramp_windows  = build_ramp_windows('hot_ramp',  qualified)
cold_ramp_windows = build_ramp_windows('cold_ramp', qualified)

t_vals = list(range(-RAMP_LOOKBACK, RAMP_LOOKAHEAD + 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, windows, stype, color, title in [
    (axes[0], hot_ramp_windows,  'hot',  'seagreen',  'Hot Ramp Signal'),
    (axes[1], cold_ramp_windows, 'cold', 'firebrick', 'Cold Ramp Signal'),
]:
    sub = windows[t_vals]
    mean_traj = sub.mean()
    sem_traj  = sub.sem()
    ax.plot(t_vals, mean_traj.values, color=color, linewidth=2.5)
    ax.fill_between(t_vals,
                    (mean_traj - sem_traj).values,
                    (mean_traj + sem_traj).values,
                    color=color, alpha=0.2)
    ax.axvline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.7, label='Signal day (t=0)')
    ax.axhline(0,           color='black', linewidth=0.7, linestyle=':', alpha=0.5)
    ax.axhline( STREAK_THRESHOLD, color=color, linewidth=0.8, linestyle=':', alpha=0.6, label='Streak threshold')
    ax.axhline(-STREAK_THRESHOLD, color=color, linewidth=0.8, linestyle=':', alpha=0.6)
    ax.set_title(f'{title}  (n={len(windows):,} signals)')
    ax.set_xlabel('Games relative to signal day')
    ax.set_ylabel('Mean deviation from season avg')
    ax.legend(fontsize=8)

fig.suptitle(f'Average Trajectory After Early Ramp Signal (slope > {SLOPE_THRESHOLD}/game)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 13. Validate Ramp Signal — Hit Rate and Precision vs Slope Threshold
# A signal "hits" if a qualifying streak starts within LEAD_WINDOW games.

def compute_hit_rate(ramp_col, streak_type, slope_col, slope_thresh):
    """Fraction of ramp signals that are followed by a qualifying streak within LEAD_WINDOW games."""
    streak_starts = (
        streak_records[
            (streak_records['streak_type'] == streak_type) &
            (streak_records['pos_in_streak'] == 1)
        ][['espn_player_id', 'game_idx']]
        .rename(columns={'game_idx': 'streak_start'})
    )

    signals = qualified[
        (qualified['deviation'].between(0, STREAK_THRESHOLD) if streak_type == 'hot'
         else qualified['deviation'].between(-STREAK_THRESHOLD, 0)) &
        (qualified[slope_col].abs() > slope_thresh) &
        (np.sign(qualified[slope_col]) == (1 if streak_type == 'hot' else -1))
    ][['espn_player_id', 'game_idx']].copy()

    if signals.empty:
        return np.nan, 0

    merged = signals.merge(streak_starts, on='espn_player_id', how='left')
    merged['gap'] = merged['streak_start'] - merged['game_idx']
    hit = merged[(merged['gap'] > 0) & (merged['gap'] <= LEAD_WINDOW)]
    hit_flags = signals.merge(
        hit.groupby(['espn_player_id', 'game_idx'])['gap'].min().reset_index(),
        on=['espn_player_id', 'game_idx'], how='left'
    )['gap'].notna()

    return hit_flags.mean(), len(signals)

# Sweep slope thresholds
thresholds = np.linspace(0.01, 0.10, 25)
results = {}
for thresh in thresholds:
    hot_pr,  hot_n  = compute_hit_rate('hot_ramp',  'hot',  'dev_slope', thresh)
    cold_pr, cold_n = compute_hit_rate('cold_ramp', 'cold', 'dev_slope', thresh)
    results[thresh] = {'hot_precision': hot_pr,  'hot_n': hot_n,
                       'cold_precision': cold_pr, 'cold_n': cold_n}

res_df = pd.DataFrame(results).T

# Baseline: random game hit rate
hot_starts  = streak_records[(streak_records['streak_type']=='hot')  & (streak_records['pos_in_streak']==1)]
cold_starts = streak_records[(streak_records['streak_type']=='cold') & (streak_records['pos_in_streak']==1)]
baseline_hot  = len(hot_starts)  / len(qualified) * LEAD_WINDOW
baseline_cold = len(cold_starts) / len(qualified) * LEAD_WINDOW

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

for ax, stype, color, baseline in [
    (axes[0], 'hot',  'seagreen',  baseline_hot),
    (axes[1], 'cold', 'firebrick', baseline_cold),
]:
    precision = res_df[f'{stype}_precision']
    n_signals = res_df[f'{stype}_n']
    ax.plot(thresholds, precision, color=color, linewidth=2, label='Signal precision')
    ax2 = ax.twinx()
    ax2.bar(thresholds, n_signals, width=0.003, alpha=0.2, color=color, label='# signals')
    ax2.set_ylabel('# Signals', fontsize=8)
    ax.axhline(baseline, color='black', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.2f})')
    ax.set_title(f'{"Hot" if stype=="hot" else "Cold"} Ramp — Precision vs Slope Threshold')
    ax.set_xlabel('Slope Threshold (deviation/game)')
    ax.set_ylabel('Hit Rate (streak within 7 games)')
    ax.legend(fontsize=8, loc='upper left')

fig.suptitle('Early Ramp Signal Precision vs Slope Threshold', fontsize=12)
plt.tight_layout()
plt.show()

# Print summary at SLOPE_THRESHOLD
hot_pr,  hot_n  = compute_hit_rate('hot_ramp',  'hot',  'dev_slope', SLOPE_THRESHOLD)
cold_pr, cold_n = compute_hit_rate('cold_ramp', 'cold', 'dev_slope', SLOPE_THRESHOLD)
print(f"\nAt slope threshold = {SLOPE_THRESHOLD}:")
print(f"  Hot  ramp precision: {hot_pr:.1%}  (n={hot_n:,})  baseline={baseline_hot:.1%}")
print(f"  Cold ramp precision: {cold_pr:.1%}  (n={cold_n:,})  baseline={baseline_cold:.1%}")

In [ ]:
# 12. Early Ramp Signal — Compute Deviation Slope
# From the event study: deviation rises ~5-8 games before a streak officially starts.
# We compute a rolling linear-regression slope over the last RAMP_WINDOW games.
# Signal = player not yet in a streak, but deviation is trending strongly toward the threshold.

RAMP_WINDOW      = 5     # games to fit slope over
SLOPE_THRESHOLD  = 0.03  # min |slope| per game to flag as ramping
LEAD_WINDOW      = 7     # games within which a streak must start to count as a "hit"

_x = np.arange(RAMP_WINDOW, dtype=float)

def _rolling_slope(series):
    return series.rolling(RAMP_WINDOW, min_periods=RAMP_WINDOW).apply(
        lambda y: np.polyfit(_x, y, 1)[0], raw=True
    )

qualified['dev_slope'] = (
    qualified
    .sort_values(['espn_player_id', 'game_idx'])
    .groupby('espn_player_id')['deviation']
    .transform(_rolling_slope)
)

# Flag ramp games: approaching threshold from neutral territory
qualified['hot_ramp'] = (
    (qualified['deviation'] > 0) &
    (qualified['deviation'] < STREAK_THRESHOLD) &
    (qualified['dev_slope'] > SLOPE_THRESHOLD)
)
qualified['cold_ramp'] = (
    (qualified['deviation'] < 0) &
    (qualified['deviation'] > -STREAK_THRESHOLD) &
    (qualified['dev_slope'] < -SLOPE_THRESHOLD)
)

print(f"Hot ramp signals  : {qualified['hot_ramp'].sum()}")
print(f"Cold ramp signals : {qualified['cold_ramp'].sum()}")
print(f"Total games       : {len(qualified)}")

In [ ]:
# 10. Survival Curve
# P(streak still alive at game N | streak has started) for hot vs cold.
# A fast-decaying curve = streaks tend to end early.
# A slow-decaying curve = once a streak starts it tends to persist.

max_len = streak_lengths['streak_length'].max()

fig, ax = plt.subplots(figsize=(10, 5))

for stype, color, label in [('hot', 'seagreen', 'Hot'), ('cold', 'firebrick', 'Cold')]:
    lengths = streak_lengths[streak_lengths['streak_type'] == stype]['streak_length'].values
    n_total = len(lengths)
    survival = [np.mean(lengths >= n) for n in range(MIN_STREAK_LEN, max_len + 1)]
    x = range(MIN_STREAK_LEN, max_len + 1)
    ax.plot(x, survival, color=color, linewidth=2, label=label)
    ax.fill_between(x, survival, alpha=0.12, color=color)

ax.axhline(0.5, color='black', linestyle='--', linewidth=0.8, alpha=0.5, label='50% survival')
ax.set_title('Streak Survival Curve — P(streak lasts ≥ N games)')
ax.set_xlabel('Streak Length (games)')
ax.set_ylabel('Proportion of Streaks Surviving')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 8. Define and Label Streaks
# A streak = min_length+ consecutive games where deviation stays above/below threshold.
# Each game in a qualifying streak is labelled: start (first 3 games), end (last 3), mid (rest).

STREAK_THRESHOLD = 0.25   # deviation units above/below season mean
MIN_STREAK_LEN   = 5      # minimum games to count as a streak
PHASE_GAMES      = 3      # how many games define the start/end phase

def find_streaks(pid, player_df):
    """Return streak records for a single player. pid passed explicitly to avoid include_groups issue."""
    dev = player_df['deviation'].values
    game_idx = player_df['game_idx'].values
    labels = np.where(dev > STREAK_THRESHOLD, 1, np.where(dev < -STREAK_THRESHOLD, -1, 0))

    records = []
    streak_id = 0
    i = 0
    while i < len(labels):
        if labels[i] != 0:
            j = i
            while j < len(labels) and labels[j] == labels[i]:
                j += 1
            length = j - i
            if length >= MIN_STREAK_LEN:
                stype = 'hot' if labels[i] == 1 else 'cold'
                for k in range(length):
                    records.append({
                        'espn_player_id': pid,
                        'game_idx':       int(game_idx[i + k]),
                        'streak_id':      f"{pid}_{streak_id}",
                        'streak_type':    stype,
                        'pos_in_streak':  k + 1,
                        'streak_length':  length,
                    })
                streak_id += 1
            i = j
        else:
            i += 1
    return pd.DataFrame(records)

sorted_q = qualified.sort_values(['espn_player_id', 'game_idx'])
streak_records = pd.concat(
    [find_streaks(pid, grp) for pid, grp in sorted_q.groupby('espn_player_id')],
    ignore_index=True
)

def assign_phase(row, phase_games=PHASE_GAMES):
    if row['pos_in_streak'] <= phase_games:
        return 'start'
    if row['pos_in_streak'] > row['streak_length'] - phase_games:
        return 'end'
    return 'mid'

streak_records['phase'] = streak_records.apply(assign_phase, axis=1)

streak_records = streak_records.merge(
    qualified[['espn_player_id', 'game_idx', 'deviation', 'playerName']],
    on=['espn_player_id', 'game_idx'], how='left'
)

print(f"Total qualifying streaks : {streak_records['streak_id'].nunique()}")
print(streak_records.groupby(['streak_type', 'phase']).size().unstack(fill_value=0))